# MVSR – Vorbereitung Einheit 5
## Pose Estimation: PnP, Optimierung, RANSAC und ICP

**Bearbeitungszeit:** ca. 45–60 Minuten

Dieses Notebook bereitet die Inhalte der fünften MVSR-Einheit praktisch vor. Aus den vorherigen Einheiten kennen Sie bereits lokale Bildmerkmale und Feature Matching. Nun folgt der nächste Schritt:

> **Wie wird aus Korrespondenzen eine räumliche Objektpose?**

Der praktische Schwerpunkt liegt auf den geometrischen Verfahren:

- 2D–3D-Korrespondenzen,
- Perspective-n-Point (PnP),
- Reprojektion und Reprojektionsfehler,
- lokale Pose-Optimierung,
- robuste Pose-Schätzung mit RANSAC,
- Registrierung von 3D-Punktwolken mit ICP.

Gelernte Verfahren wie **Render-and-Compare**, **FoundationPose** und **Category-Level Pose Estimation** werden am Ende eingeordnet. Eine vollständige Ausführung dieser Modelle ist für dieses Vorbereitungsnotebook bewusst nicht vorgesehen.

Für dieses Binder-Notebook wird **C++17 mit OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- erklären, was eine 6D-Pose beschreibt,
- 2D-Bildpunkte bekannten 3D-Objektpunkten zuordnen,
- mit `cv::solvePnP()` Rotation und Translation bestimmen,
- die Rechenkette von 3D-Punkt über Kamerakoordinaten bis zum Bildpunkt nachvollziehen,
- die geschätzte Objektpose als Koordinatensystem im Kamerabild visualisieren,
- einen Reprojektionsfehler berechnen,
- die Bedeutung des Projektions-Jacobians erklären,
- eine initiale Pose mit Levenberg–Marquardt verfeinern,
- mit `cv::solvePnPRansac()` Outlier robust behandeln,
- den Grundablauf von Point-to-Point ICP nachvollziehen,
- Grenzen von ICP und die Rolle einer guten Initialpose erklären,
- klassische und gelernte Pose-Estimation-Ansätze einordnen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook wird über **MyBinder** direkt im Browser ausgeführt. Der vorhandene C++-Code kann unmittelbar in den Notebook-Zellen bearbeitet und ausgeführt werden; eine zusätzliche lokale Installation ist dafür nicht erforderlich.

Arbeiten Sie das Notebook möglichst **von oben nach unten** durch. Spätere Zellen verwenden Variablen und Ergebnisse aus früheren Abschnitten.

Bei Aufgaben mit `TODO` sollen Sie Parameter selbstständig verändern und deren Einfluss auf das Ergebnis beobachten.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Bereits deklarierte Variablen können daher bei erneutem Ausführen derselben Zelle zu einer Fehlermeldung führen. Übungszellen, die häufig erneut ausgeführt werden, sind deshalb teilweise in lokalen Lambda-Funktionen gekapselt.

> **Wichtig:** Änderungen innerhalb einer Binder-Sitzung werden nicht dauerhaft im GitHub-Repository gespeichert.

## 0. Setup

Die Binder-Umgebung ist bereits mit **xeus-cling** und **OpenCV 4.6** vorbereitet.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind spezifisch für den interaktiven C++-Kernel. In einem normalen lokalen C++-Projekt wird OpenCV über das Build-System, z.B. mit CMake, eingebunden und gelinkt.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <fstream>
#include <cstdlib>
#include <algorithm>
#include <limits>
#include <numeric>

#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")
#pragma cling load("opencv_features2d")
#pragma cling load("opencv_calib3d")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Bilddarstellung

In einem lokalen C++-Programm würde für die Darstellung typischerweise `cv::imshow()` und `cv::waitKey()` verwendet.

Da MyBinder im Browser läuft, wird hier wieder die aus den vorherigen C++-Notebooks bekannte Rich-Display-Hilfsfunktion verwendet.

> **Binder-/Notebook-Hinweis:** `show_image(...)` muss als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

### Hilfsfunktion zum Laden externer Testbilder

Die Pose-Beispiele verwenden offizielle OpenCV-Testbilder. In Colab wurden diese mit Python bzw. `wget` geladen.

> **Binder-/Notebook-Hinweis:** Hier erfolgt der Download über das Kommandozeilenprogramm `wget`. In einem normalen lokalen Projekt würden die Testbilder typischerweise direkt im Projektordner abgelegt und mit `cv::imread()` geladen.

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \"" + url + "\" -O \"" + output_file + "\"";

    int result = std::system(command.c_str());

    if (result != 0)
    {
        std::cerr << "Download fehlgeschlagen: "
                  << url << std::endl;
        return false;
    }

    return true;
}

### Hilfsfunktion für den mittleren Reprojektionsfehler

In [ ]:
double mean_reprojection_error(
    const std::vector<cv::Point3f>& object_points,
    const std::vector<cv::Point2f>& image_points,
    const cv::Mat& rvec,
    const cv::Mat& tvec,
    const cv::Mat& camera_matrix,
    const cv::Mat& dist_coeffs,
    std::vector<cv::Point2f>* projected_output = nullptr
)
{
    std::vector<cv::Point2f> projected_points;

    cv::projectPoints(
        object_points,
        rvec,
        tvec,
        camera_matrix,
        dist_coeffs,
        projected_points
    );

    double error_sum = 0.0;

    for (std::size_t i = 0; i < image_points.size(); ++i)
    {
        error_sum += cv::norm(
            image_points[i] - projected_points[i]
        );
    }

    if (projected_output != nullptr)
    {
        *projected_output = projected_points;
    }

    return error_sum / image_points.size();
}

## 1. Von Merkmalen zur Pose

Aus den vorherigen Einheiten kennen wir bereits:

**Bild → Merkmale → Matching → Korrespondenzen**

Feature Matching liefert zunächst **2D-Korrespondenzen im Bild**. Für eine räumliche Objektpose benötigen wir zusätzlich bekannte 3D-Information.

Daher betrachten wir jetzt Zuordnungen der Form

$\mathbf{P}_i \leftrightarrow \mathbf{p}_i$

mit

- $\mathbf{P}_i$: bekannter 3D-Punkt im Objektkoordinatensystem,
- $\mathbf{p}_i$: zugehöriger 2D-Punkt im Kamerabild.

Aus mehreren solchen 2D–3D-Korrespondenzen wird die Objektpose relativ zur Kamera bestimmt.

### Was ist eine 6D-Pose?

Eine Pose besteht aus

- Translation $\mathbf{t}=[x,y,z]^T$,
- Rotation $\mathbf{R}$.

Gemeinsam können beide als homogene Transformation geschrieben werden:

${}^{c}\mathbf{T}_{o}
=
\begin{bmatrix}
\mathbf{R} & \mathbf{t}\\
\mathbf{0}^{T} & 1
\end{bmatrix}$

${}^{c}\mathbf{T}_{o}$ transformiert Punkte aus dem **Objektkoordinatensystem** in das **Kamerakoordinatensystem**.

## 2. Reales Testbild und bekannte 3D-Struktur

Für die praktische Untersuchung verwenden wir reale Schachbrettaufnahmen aus den offiziellen OpenCV-Beispieldaten.

Das Muster besitzt **7 × 6 innere Eckpunkte**. Die reale Seitenlänge eines Quadrats ist für diese Beispielbilder nicht angegeben. Deshalb setzen wir

`square_size = 1.0`.

Die Translation wird damit in **Schachbrett-Quadratlängen** angegeben.

In [ ]:
std::system("mkdir -p pose_data");

std::string base_url =
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data";

std::vector<std::string> image_files;

for (int i = 1; i <= 14; ++i)
{
    std::ostringstream filename_stream;
    filename_stream
        << "left"
        << std::setw(2)
        << std::setfill('0')
        << i
        << ".jpg";

    std::string filename =
        filename_stream.str();

    std::string target =
        "pose_data/" + filename;

    if (download_file(
        base_url + "/" + filename,
        target
    ))
    {
        image_files.push_back(target);
    }
}

std::cout << "Geladene Bilder: "
          << image_files.size()
          << std::endl;

In [ ]:
cv::Mat image =
    cv::imread(
        "pose_data/left01.jpg",
        cv::IMREAD_COLOR
    );

if (image.empty())
{
    std::cerr << "Testbild konnte nicht geladen werden."
              << std::endl;
}

cv::Mat gray;

cv::cvtColor(
    image,
    gray,
    cv::COLOR_BGR2GRAY
);

In [ ]:
show_image(image)

## 3. 2D–3D-Korrespondenzen

Die bekannten 3D-Punkte des Schachbretts liegen auf einer Ebene:

$\mathbf{P}_i =
\begin{bmatrix}
X_i\\
Y_i\\
0
\end{bmatrix}$

Die zugehörigen Bildpunkte werden mit `cv::findChessboardCorners()` erkannt und anschließend mit `cv::cornerSubPix()` genauer lokalisiert.

Damit erhalten wir die für PnP benötigten 2D–3D-Korrespondenzen.

In [ ]:
cv::Size pattern_size(7, 6);
float square_size = 1.0f;

std::vector<cv::Point3f> object_points;

for (int y = 0; y < pattern_size.height; ++y)
{
    for (int x = 0; x < pattern_size.width; ++x)
    {
        object_points.emplace_back(
            x * square_size,
            y * square_size,
            0.0f
        );
    }
}

std::cout << "Anzahl 3D-Punkte: "
          << object_points.size()
          << std::endl;

std::cout << "Erste fünf 3D-Punkte:"
          << std::endl;

for (int i = 0; i < 5; ++i)
{
    std::cout << object_points[i]
              << std::endl;
}

In [ ]:
std::vector<cv::Point2f> image_points;

bool found = cv::findChessboardCorners(
    gray,
    pattern_size,
    image_points
);

if (!found)
{
    std::cerr << "Schachbrettmuster wurde nicht gefunden."
              << std::endl;
}

cv::TermCriteria criteria_subpix(
    cv::TermCriteria::EPS
        + cv::TermCriteria::MAX_ITER,
    30,
    0.001
);

if (found)
{
    cv::cornerSubPix(
        gray,
        image_points,
        cv::Size(11, 11),
        cv::Size(-1, -1),
        criteria_subpix
    );
}

cv::Mat image_corners =
    image.clone();

cv::drawChessboardCorners(
    image_corners,
    pattern_size,
    image_points,
    found
);

std::cout << "Anzahl 2D-Punkte: "
          << image_points.size()
          << std::endl;

In [ ]:
show_image(image_corners)

## 4. Intrinsische Kameraparameter

PnP benötigt zusätzlich die intrinsische Kameramatrix

$\mathbf{K}=
\begin{bmatrix}
f_x & 0 & c_x\\
0 & f_y & c_y\\
0 & 0 & 1
\end{bmatrix}$.

In einer realen Anwendung stammen $\mathbf{K}$ und die Verzeichnungsparameter aus einer zuvor durchgeführten **Kamerakalibrierung**.

Für dieses Notebook bestimmen wir sie einmal aus mehreren OpenCV-Schachbrettbildern. Die Kalibrierung selbst ist hier nur Vorbereitung; anschließend behandeln wir die Parameter als gegeben.

In [ ]:
std::vector<std::vector<cv::Point3f>>
    calib_object_points;

std::vector<std::vector<cv::Point2f>>
    calib_image_points;

for (const auto& path : image_files)
{
    cv::Mat img_calib =
        cv::imread(path, cv::IMREAD_COLOR);

    if (img_calib.empty())
    {
        continue;
    }

    cv::Mat gray_calib;

    cv::cvtColor(
        img_calib,
        gray_calib,
        cv::COLOR_BGR2GRAY
    );

    std::vector<cv::Point2f> corners_calib;

    bool found_calib =
        cv::findChessboardCorners(
            gray_calib,
            pattern_size,
            corners_calib
        );

    if (!found_calib)
    {
        continue;
    }

    cv::cornerSubPix(
        gray_calib,
        corners_calib,
        cv::Size(11, 11),
        cv::Size(-1, -1),
        criteria_subpix
    );

    calib_object_points.push_back(
        object_points
    );

    calib_image_points.push_back(
        corners_calib
    );
}

std::cout
    << "Für Kalibrierung verwendete Aufnahmen: "
    << calib_image_points.size()
    << std::endl;

In [ ]:
cv::Mat camera_matrix;
cv::Mat dist_coeffs;

std::vector<cv::Mat> calibration_rvecs;
std::vector<cv::Mat> calibration_tvecs;

double calibration_error =
    cv::calibrateCamera(
        calib_object_points,
        calib_image_points,
        gray.size(),
        camera_matrix,
        dist_coeffs,
        calibration_rvecs,
        calibration_tvecs
    );

std::cout << "Kameramatrix K:"
          << std::endl
          << camera_matrix
          << std::endl;

std::cout << "\nVerzeichnungsparameter:"
          << std::endl
          << dist_coeffs
          << std::endl;

std::cout << "\nKalibrierungs-RMS-Fehler: "
          << calibration_error
          << std::endl;

## 5. Perspective-n-Point

Perspective-n-Point bestimmt die Pose aus

- bekannten 3D-Punkten,
- zugehörigen 2D-Bildpunkten,
- intrinsischen Kameraparametern.

Gesucht sind $\mathbf{R}$ und $\mathbf{t}$:

$\text{PnP}(\mathbf{P}_i,\mathbf{p}_i,\mathbf{K})
\rightarrow
\mathbf{R},\mathbf{t}$

OpenCV liefert mit `cv::solvePnP()` zunächst

- `rvec`: Rotationsvektor,
- `tvec`: Translationsvektor.

Mit `cv::Rodrigues()` kann `rvec` in eine Rotationsmatrix umgewandelt werden.

Weitere Informationen: [OpenCV-4.6-Dokumentation zu `solvePnP()`](https://docs.opencv.org/4.6.0/d5/d1f/calib3d_solvePnP.html).

In [ ]:
cv::Mat rvec;
cv::Mat tvec;

bool success = cv::solvePnP(
    object_points,
    image_points,
    camera_matrix,
    dist_coeffs,
    rvec,
    tvec,
    false,
    cv::SOLVEPNP_ITERATIVE
);

cv::Mat rotation_matrix;

cv::Rodrigues(
    rvec,
    rotation_matrix
);

cv::Mat T_c_o =
    cv::Mat::eye(
        4,
        4,
        CV_64F
    );

rotation_matrix.copyTo(
    T_c_o(
        cv::Rect(0, 0, 3, 3)
    )
);

tvec.copyTo(
    T_c_o(
        cv::Rect(3, 0, 1, 3)
    )
);

std::cout << "PnP erfolgreich: "
          << std::boolalpha
          << success
          << std::endl;

std::cout << "\nrvec:"
          << std::endl
          << rvec
          << std::endl;

std::cout << "\ntvec:"
          << std::endl
          << tvec
          << std::endl;

std::cout << "\nT_c_o:"
          << std::endl
          << T_c_o
          << std::endl;

### Objektkoordinatensystem visualisieren

Die geschätzte Pose lässt sich direkt im Kamerabild darstellen. Dazu projizieren wir das **Koordinatensystem des Objekts** mit der berechneten Pose in das Bild.

Der Ursprung liegt beim ersten Schachbrett-Eckpunkt $\mathbf{P}_0=[0,0,0]^T$.

OpenCV zeichnet die Achsen mit

- **X-Achse:** rot,
- **Y-Achse:** grün,
- **Z-Achse:** blau.

Damit wird unmittelbar sichtbar, **wo sich das Objekt relativ zur Kamera befindet und wie es orientiert ist**.

In [ ]:
float axis_length =
    3.0f * square_size;

cv::Mat image_pose =
    image.clone();

cv::drawFrameAxes(
    image_pose,
    camera_matrix,
    dist_coeffs,
    rvec,
    tvec,
    axis_length,
    3
);

In [ ]:
show_image(image_pose)

## 6. PnP-Rechenkette nachvollziehen

PnP kann als Optimierungsproblem verstanden werden:

$\mathbf{P}_i
\rightarrow
\mathbf{P}_i^c
\rightarrow
\hat{\mathbf{p}}_i
\rightarrow
\mathbf{e}_i$

Zuerst wird ein Objektpunkt in das Kamerakoordinatensystem transformiert:

$\mathbf{P}_i^c
=
\mathbf{R}\mathbf{P}_i+\mathbf{t}$

Anschließend erfolgt die perspektivische Projektion:

$\hat{u}_i =
f_x\frac{X_i^c}{Z_i^c}+c_x$

$\hat{v}_i =
f_y\frac{Y_i^c}{Z_i^c}+c_y$.

Für diese Herleitung ignorieren wir zunächst die Linsenverzeichnung.

In [ ]:
int point_index = 20;

cv::Point3f P_object =
    object_points[point_index];

cv::Mat P_object_mat =
    (cv::Mat_<double>(3, 1)
        << P_object.x,
           P_object.y,
           P_object.z);

cv::Mat P_camera_mat =
    rotation_matrix
        * P_object_mat
    + tvec;

double Xc =
    P_camera_mat.at<double>(0);

double Yc =
    P_camera_mat.at<double>(1);

double Zc =
    P_camera_mat.at<double>(2);

double fx =
    camera_matrix.at<double>(0, 0);

double fy =
    camera_matrix.at<double>(1, 1);

double cx =
    camera_matrix.at<double>(0, 2);

double cy =
    camera_matrix.at<double>(1, 2);

double u_manual =
    fx * Xc / Zc + cx;

double v_manual =
    fy * Yc / Zc + cy;

std::vector<cv::Point3f>
    one_object_point = {P_object};

std::vector<cv::Point2f>
    projected_no_distortion;

cv::Mat zero_dist =
    cv::Mat::zeros(
        dist_coeffs.size(),
        dist_coeffs.type()
    );

cv::projectPoints(
    one_object_point,
    rvec,
    tvec,
    camera_matrix,
    zero_dist,
    projected_no_distortion
);

std::cout << "Punkt im Objektkoordinatensystem: "
          << P_object
          << std::endl;

std::cout << "Punkt im Kamerakoordinatensystem: "
          << cv::Point3d(Xc, Yc, Zc)
          << std::endl;

std::cout << "Manuelle Projektion: "
          << cv::Point2d(u_manual, v_manual)
          << std::endl;

std::cout << "cv::projectPoints ohne Verzeichnung: "
          << projected_no_distortion[0]
          << std::endl;

Die manuelle Projektion und `cv::projectPoints()` liefern ohne Verzeichnung praktisch dasselbe Ergebnis.

Damit wird sichtbar, was PnP intern wiederholt benötigt: **eine Pose erzeugt vorhergesagte Bildpunkte**.

## 7. Reprojektionsfehler

Die Kamera misst einen Bildpunkt $\mathbf{p}_i$. Die aktuelle Pose erzeugt einen vorhergesagten Bildpunkt $\hat{\mathbf{p}}_i$.

Der Fehler lautet:

$\mathbf{e}_i=
\mathbf{p}_i-\hat{\mathbf{p}}_i$

und als quadratisches Optimierungsproblem:

$E=
\sum_i
\|\mathbf{e}_i\|^2$.

Ein einfaches Qualitätsmaß ist der mittlere Reprojektionsfehler in Pixeln.

In [ ]:
std::vector<cv::Point2f>
    projected_points;

double mean_error =
    mean_reprojection_error(
        object_points,
        image_points,
        rvec,
        tvec,
        camera_matrix,
        dist_coeffs,
        &projected_points
    );

std::cout
    << "Mittlerer Reprojektionsfehler: "
    << mean_error
    << " Pixel"
    << std::endl;

In [ ]:
cv::Mat image_reprojection =
    image.clone();

for (std::size_t i = 0;
     i < image_points.size();
     ++i)
{
    cv::circle(
        image_reprojection,
        image_points[i],
        3,
        cv::Scalar(0, 255, 0),
        -1
    );

    cv::circle(
        image_reprojection,
        projected_points[i],
        2,
        cv::Scalar(0, 0, 255),
        -1
    );
}

In [ ]:
show_image(image_reprojection)

## 8. Lokale Idee der Pose-Optimierung

Die Projektion ist nichtlinear. Für eine kleine Änderung kann sie lokal linearisiert werden.

Für einen Punkt im Kamerakoordinatensystem $\mathbf{P}^c=[X,Y,Z]^T$ lautet der Projektions-Jacobian:

$\mathbf{J}_{\pi}
=
\begin{bmatrix}
\frac{f_x}{Z} & 0 & -\frac{f_xX}{Z^2}\\
0 & \frac{f_y}{Z} & -\frac{f_yY}{Z^2}
\end{bmatrix}$

Für eine kleine Poseänderung

$\Delta\boldsymbol{\xi}
=
[\Delta r_x,\Delta r_y,\Delta r_z,
\Delta t_x,\Delta t_y,\Delta t_z]^T$

ergibt sich:

$\mathbf{J}_i
=
\mathbf{J}_{\pi}
\begin{bmatrix}
-[\mathbf{P}_i^c]_{\times} & \mathbf{I}
\end{bmatrix}$

mit $\mathbf{J}_i\in\mathbb{R}^{2\times6}$.

In [ ]:
cv::Mat J_pi =
    (cv::Mat_<double>(2, 3)
        << fx / Zc,
           0.0,
           -fx * Xc / (Zc * Zc),

           0.0,
           fy / Zc,
           -fy * Yc / (Zc * Zc));

cv::Mat skew_P =
    (cv::Mat_<double>(3, 3)
        << 0.0, -Zc,  Yc,
           Zc,   0.0, -Xc,
          -Yc,   Xc,  0.0);

cv::Mat pose_derivative =
    cv::Mat::zeros(
        3,
        6,
        CV_64F
    );

cv::Mat minus_skew_P =
    -skew_P;

cv::Mat identity =
    cv::Mat::eye(
        3,
        3,
        CV_64F
    );

minus_skew_P.copyTo(
    pose_derivative(
        cv::Rect(0, 0, 3, 3)
    )
);

identity.copyTo(
    pose_derivative(
        cv::Rect(3, 0, 3, 3)
    )
);

cv::Mat J_pose =
    J_pi * pose_derivative;

std::cout << "Projektions-Jacobian J_pi:"
          << std::endl
          << J_pi
          << std::endl;

std::cout << "\nPose-Jacobian J_i:"
          << std::endl
          << J_pose
          << std::endl;

std::cout << "\nForm von J_i: "
          << J_pose.rows
          << " x "
          << J_pose.cols
          << std::endl;

### Intuition des Projektions-Jacobians

Wir verändern den 3D-Punkt nur sehr wenig und vergleichen:

- die **exakte neue Projektion**,
- die mit $\mathbf{J}_{\pi}$ vorhergesagte Bildverschiebung.

Für ausreichend kleine Änderungen sollten beide Ergebnisse ähnlich sein.

In [ ]:
cv::Mat delta_P =
    (cv::Mat_<double>(3, 1)
        << 0.01,
          -0.01,
           0.0);

cv::Mat pixel_before =
    (cv::Mat_<double>(2, 1)
        << fx * Xc / Zc + cx,
           fy * Yc / Zc + cy);

cv::Mat P_camera_new =
    P_camera_mat + delta_P;

double X_new =
    P_camera_new.at<double>(0);

double Y_new =
    P_camera_new.at<double>(1);

double Z_new =
    P_camera_new.at<double>(2);

cv::Mat pixel_after =
    (cv::Mat_<double>(2, 1)
        << fx * X_new / Z_new + cx,
           fy * Y_new / Z_new + cy);

cv::Mat delta_pixel_exact =
    pixel_after - pixel_before;

cv::Mat delta_pixel_linear =
    J_pi * delta_P;

std::cout << "Exakte Bildverschiebung:"
          << std::endl
          << delta_pixel_exact
          << std::endl;

std::cout << "\nLineare Näherung über J_pi:"
          << std::endl
          << delta_pixel_linear
          << std::endl;

## 9. Pose-Verfeinerung mit Levenberg–Marquardt

Ein iteratives PnP-Verfahren verbessert eine Pose schrittweise:

**aktuelle Pose → Reprojektion → Fehler → Jacobian → Update → neue Pose**

Eine typische lokale Optimierung verwendet Levenberg–Marquardt:

$(\mathbf{J}^T\mathbf{J}+\lambda\mathbf{I})
\Delta\boldsymbol{\xi}
=
\mathbf{J}^T\mathbf{e}$

OpenCV stellt mit `cv::solvePnPRefineLM()` direkt eine solche Pose-Verfeinerung bereit.

Wir verschlechtern zunächst die bereits gute Pose künstlich und lassen sie anschließend wieder verfeinern.

In [ ]:
cv::Mat rvec_initial =
    rvec.clone();

cv::Mat tvec_initial =
    tvec.clone();

rvec_initial +=
    (cv::Mat_<double>(3, 1)
        << 0.08,
          -0.05,
           0.04);

tvec_initial +=
    (cv::Mat_<double>(3, 1)
        << 0.25,
          -0.20,
           0.50);

double error_initial =
    mean_reprojection_error(
        object_points,
        image_points,
        rvec_initial,
        tvec_initial,
        camera_matrix,
        dist_coeffs
    );

cv::Mat rvec_refined =
    rvec_initial.clone();

cv::Mat tvec_refined =
    tvec_initial.clone();

cv::TermCriteria criteria_lm(
    cv::TermCriteria::EPS
        + cv::TermCriteria::COUNT,
    30,
    1e-6
);

cv::solvePnPRefineLM(
    object_points,
    image_points,
    camera_matrix,
    dist_coeffs,
    rvec_refined,
    tvec_refined,
    criteria_lm
);

double error_refined =
    mean_reprojection_error(
        object_points,
        image_points,
        rvec_refined,
        tvec_refined,
        camera_matrix,
        dist_coeffs
    );

std::cout
    << "Fehler der gestörten Startpose: "
    << error_initial
    << " Pixel"
    << std::endl;

std::cout
    << "Fehler nach LM-Refinement: "
    << error_refined
    << " Pixel"
    << std::endl;

### Mini-Aufgabe: Initialpose verändern

Verändern Sie die künstliche Störung von Rotation und Translation.

Beobachten Sie:

- Wie groß darf die Abweichung sein, bevor die lokale Optimierung Probleme bekommt?
- Warum ist eine gute Initialpose für lokale Optimierungsverfahren hilfreich?

In [ ]:
[]()
{
    // TODO: Stärke der Startabweichung verändern

    cv::Mat rotation_offset_test =
        (cv::Mat_<double>(3, 1)
            << 0.15,
              -0.10,
               0.08);

    cv::Mat translation_offset_test =
        (cv::Mat_<double>(3, 1)
            << 0.50,
              -0.30,
               1.00);

    cv::Mat rvec_start_test =
        rvec + rotation_offset_test;

    cv::Mat tvec_start_test =
        tvec + translation_offset_test;

    double error_start_test =
        mean_reprojection_error(
            object_points,
            image_points,
            rvec_start_test,
            tvec_start_test,
            camera_matrix,
            dist_coeffs
        );

    cv::Mat rvec_lm_test =
        rvec_start_test.clone();

    cv::Mat tvec_lm_test =
        tvec_start_test.clone();

    cv::solvePnPRefineLM(
        object_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        rvec_lm_test,
        tvec_lm_test,
        criteria_lm
    );

    double error_lm_test =
        mean_reprojection_error(
            object_points,
            image_points,
            rvec_lm_test,
            tvec_lm_test,
            camera_matrix,
            dist_coeffs
        );

    std::cout
        << "Startfehler: "
        << error_start_test
        << " Pixel"
        << std::endl;

    std::cout
        << "Fehler nach LM: "
        << error_lm_test
        << " Pixel"
        << std::endl;
}()

## 10. Robustheit durch RANSAC

Bei realem Feature Matching sind Korrespondenzen nicht immer korrekt.

- **Inlier:** 2D- und 3D-Punkt gehören tatsächlich zusammen.
- **Outlier:** fehlerhafte Zuordnung.

Bereits wenige starke Outlier können eine Pose deutlich verfälschen.

RANSAC arbeitet vereinfacht nach folgendem Prinzip:

1. kleine Teilmenge der Korrespondenzen auswählen,
2. Pose-Hypothese berechnen,
3. alle Punkte reprojizieren,
4. Inlier anhand des Reprojektionsfehlers bestimmen,
5. viele Hypothesen testen und die mit dem größten Konsens wählen.

Wir simulieren falsche Matches, indem wir einige erkannte Bildpunkte absichtlich verschieben.

In [ ]:
std::vector<cv::Point2f>
    image_points_outliers =
        image_points;

std::vector<int>
    outlier_indices = {
        5,
        18,
        31,
        38
    };

std::vector<cv::Point2f>
    outlier_offsets = {
        cv::Point2f( 80.0f, -50.0f),
        cv::Point2f(-70.0f,  65.0f),
        cv::Point2f( 55.0f,  75.0f),
        cv::Point2f(-90.0f, -40.0f)
    };

for (std::size_t i = 0;
     i < outlier_indices.size();
     ++i)
{
    image_points_outliers[
        outlier_indices[i]
    ] += outlier_offsets[i];
}

cv::Mat image_outliers =
    image.clone();

for (std::size_t i = 0;
     i < image_points_outliers.size();
     ++i)
{
    bool is_outlier =
        std::find(
            outlier_indices.begin(),
            outlier_indices.end(),
            static_cast<int>(i)
        ) != outlier_indices.end();

    cv::circle(
        image_outliers,
        image_points_outliers[i],
        is_outlier ? 6 : 3,
        is_outlier
            ? cv::Scalar(0, 0, 255)
            : cv::Scalar(0, 255, 0),
        -1
    );
}

In [ ]:
show_image(image_outliers)

### Normales PnP mit Outliern

Zunächst behandeln wir alle Korrespondenzen fälschlicherweise als korrekt.

In [ ]:
cv::Mat rvec_outlier;
cv::Mat tvec_outlier;

bool success_outlier =
    cv::solvePnP(
        object_points,
        image_points_outliers,
        camera_matrix,
        dist_coeffs,
        rvec_outlier,
        tvec_outlier,
        false,
        cv::SOLVEPNP_ITERATIVE
    );

double error_pnp_outliers =
    mean_reprojection_error(
        object_points,
        image_points,
        rvec_outlier,
        tvec_outlier,
        camera_matrix,
        dist_coeffs
    );

std::cout
    << "Fehler von PnP mit Outliern: "
    << error_pnp_outliers
    << " Pixel"
    << std::endl;

### PnP mit RANSAC

Der Parameter `reprojectionError` legt fest, wie groß der Reprojektionsfehler maximal sein darf, damit eine Korrespondenz als Inlier gilt.

In [ ]:
cv::Mat rvec_ransac;
cv::Mat tvec_ransac;
cv::Mat inliers;

bool success_ransac =
    cv::solvePnPRansac(
        object_points,
        image_points_outliers,
        camera_matrix,
        dist_coeffs,
        rvec_ransac,
        tvec_ransac,
        false,
        100,
        3.0,
        0.99,
        inliers,
        cv::SOLVEPNP_ITERATIVE
    );

double error_ransac =
    mean_reprojection_error(
        object_points,
        image_points,
        rvec_ransac,
        tvec_ransac,
        camera_matrix,
        dist_coeffs
    );

std::cout
    << "Von RANSAC erkannte Inlier: "
    << inliers.rows
    << " von "
    << object_points.size()
    << std::endl;

std::cout
    << "Fehler der RANSAC-Pose: "
    << error_ransac
    << " Pixel"
    << std::endl;

In [ ]:
cv::Mat image_ransac =
    image.clone();

std::vector<bool>
    is_inlier(
        image_points_outliers.size(),
        false
    );

for (int i = 0;
     i < inliers.rows;
     ++i)
{
    int index =
        inliers.at<int>(i, 0);

    is_inlier[index] = true;
}

for (std::size_t i = 0;
     i < image_points_outliers.size();
     ++i)
{
    cv::circle(
        image_ransac,
        image_points_outliers[i],
        5,
        is_inlier[i]
            ? cv::Scalar(0, 255, 0)
            : cv::Scalar(0, 0, 255),
        -1
    );
}

In [ ]:
show_image(image_ransac)

### Güte einer robust geschätzten Pose

Nach RANSAC sollte nicht nur die Pose selbst betrachtet werden.

Sinnvolle Qualitätsmerkmale sind:

- Anzahl der Inlier,
- Inlier-Anteil,
- mittlerer Reprojektionsfehler,
- geometrische Plausibilität der resultierenden Pose.

In [ ]:
double inlier_ratio =
    static_cast<double>(inliers.rows)
    / object_points.size();

std::cout
    << "Inlier-Anteil: "
    << 100.0 * inlier_ratio
    << " %"
    << std::endl;

std::cout
    << "PnP mit Outliern: "
    << error_pnp_outliers
    << " px"
    << std::endl;

std::cout
    << "PnP + RANSAC: "
    << error_ransac
    << " px"
    << std::endl;

### Mini-Aufgabe: RANSAC-Parameter

Verändern Sie `reprojectionError` und `iterationsCount`.

Beobachten Sie:

- Wie verändert sich die Zahl der Inlier?
- Was passiert bei einem sehr kleinen Reprojektions-Threshold?
- Was passiert bei einem sehr großen Threshold?

In [ ]:
[]()
{
    // TODO: RANSAC-Parameter verändern

    int iterations_count_test = 100;
    float reprojection_error_test = 5.0f;
    double confidence_test = 0.99;

    cv::Mat rvec_test;
    cv::Mat tvec_test;
    cv::Mat inliers_test;

    bool success_test =
        cv::solvePnPRansac(
            object_points,
            image_points_outliers,
            camera_matrix,
            dist_coeffs,
            rvec_test,
            tvec_test,
            false,
            iterations_count_test,
            reprojection_error_test,
            confidence_test,
            inliers_test,
            cv::SOLVEPNP_ITERATIVE
        );

    if (!success_test)
    {
        std::cerr
            << "Mit diesen Parametern konnte keine Pose bestimmt werden."
            << std::endl;
        return;
    }

    double error_test =
        mean_reprojection_error(
            object_points,
            image_points,
            rvec_test,
            tvec_test,
            camera_matrix,
            dist_coeffs
        );

    std::cout
        << "Inlier: "
        << inliers_test.rows
        << " / "
        << object_points.size()
        << std::endl;

    std::cout
        << "Reprojektionsfehler: "
        << error_test
        << " Pixel"
        << std::endl;
}()

## 11. Iterative Closest Point

PnP arbeitet mit **2D–3D-Korrespondenzen**. ICP betrachtet dagegen zwei 3D-Punktmengen.

Gesucht ist eine Transformation zwischen

- bewegter Punktwolke $\mathcal{P}$,
- Zielpunktwolke $\mathcal{Q}$.

Die einfachste Variante ist **Point-to-Point ICP**:

1. zu jedem Punkt den nächsten Zielpunkt finden,
2. beste starre Transformation zwischen den Punktpaaren bestimmen,
3. Punktwolke transformieren,
4. Korrespondenzen neu bestimmen,
5. wiederholen.

Das Optimierungsproblem lautet:

$\min_{\mathbf{R},\mathbf{t}}
\sum_i
\|
\mathbf{q}_i-
(\mathbf{R}\mathbf{p}_i+\mathbf{t})
\|^2$.

Für diesen Abschnitt verwenden wir bewusst ein kleines simuliertes 3D-Modell. Dadurch lässt sich der ICP-Ablauf unabhängig von Sensorrauschen oder Segmentierungsfehlern untersuchen.

In [ ]:
std::vector<cv::Point3d>
create_asymmetric_point_cloud()
{
    std::vector<cv::Point3d> points;

    for (int ix = 0; ix < 10; ++ix)
    {
        double x =
            static_cast<double>(ix) / 9.0;

        for (int iy = 0; iy < 7; ++iy)
        {
            double y =
                0.6 * static_cast<double>(iy) / 6.0;

            points.emplace_back(
                x,
                y,
                0.0
            );
        }
    }

    for (int iy = 0; iy < 7; ++iy)
    {
        double y =
            0.6 * static_cast<double>(iy) / 6.0;

        for (int iz = 0; iz < 9; ++iz)
        {
            double z =
                0.8 * static_cast<double>(iz) / 8.0;

            points.emplace_back(
                0.0,
                y,
                z
            );
        }
    }

    for (int ix = 0; ix < 5; ++ix)
    {
        double x =
            0.55 + 0.40 * static_cast<double>(ix) / 4.0;

        for (int iz = 0; iz < 5; ++iz)
        {
            double z =
                0.35 * static_cast<double>(iz) / 4.0;

            points.emplace_back(
                x,
                0.6,
                z
            );
        }
    }

    return points;
}

In [ ]:
std::vector<cv::Point3d>
    target_cloud =
        create_asymmetric_point_cloud();

double icp_start_angle =
    12.0 * CV_PI / 180.0;

cv::Matx33d R_initial(
    std::cos(icp_start_angle), -std::sin(icp_start_angle), 0.0,
    std::sin(icp_start_angle),  std::cos(icp_start_angle), 0.0,
    0.0,                        0.0,                       1.0
);

cv::Vec3d t_initial(
    0.15,
    -0.10,
    0.08
);

std::vector<cv::Point3d>
    source_cloud;

for (const auto& point : target_cloud)
{
    cv::Vec3d p(
        point.x,
        point.y,
        point.z
    );

    cv::Vec3d transformed =
        R_initial * p
        + t_initial;

    source_cloud.emplace_back(
        transformed[0],
        transformed[1],
        transformed[2]
    );
}

std::cout << "Punkte pro Punktwolke: "
          << target_cloud.size()
          << std::endl;

### Punktwolken im Binder-Notebook darstellen

In der Python-Version kann Matplotlib direkt eine interaktive 3D-Achse erzeugen. Für das reine C++-Notebook verwenden wir stattdessen eine einfache **isometrische 2D-Projektion** der 3D-Punkte.

> **Binder-/Notebook-Hinweis:** Diese Darstellung dient nur der Visualisierung im Browser. Für eine lokale C++-Anwendung würde man für echte 3D-Darstellung typischerweise z.B. PCL, Open3D oder eine entsprechende 3D-Visualisierungsbibliothek verwenden.

In [ ]:
cv::Point2d project_isometric(
    const cv::Point3d& p
)
{
    double u =
        p.x - 0.6 * p.y;

    double v =
        0.35 * p.x
        + 0.35 * p.y
        - p.z;

    return cv::Point2d(u, v);
}

In [ ]:
cv::Mat render_point_clouds(
    const std::vector<cv::Point3d>& source,
    const std::vector<cv::Point3d>& target,
    const std::string& title = ""
)
{
    const int width = 760;
    const int height = 560;
    const int margin = 50;

    std::vector<cv::Point2d> projected_source;
    std::vector<cv::Point2d> projected_target;

    for (const auto& p : source)
    {
        projected_source.push_back(
            project_isometric(p)
        );
    }

    for (const auto& p : target)
    {
        projected_target.push_back(
            project_isometric(p)
        );
    }

    double min_u =
        std::numeric_limits<double>::max();

    double max_u =
        -std::numeric_limits<double>::max();

    double min_v =
        std::numeric_limits<double>::max();

    double max_v =
        -std::numeric_limits<double>::max();

    auto update_bounds =
        [&](const std::vector<cv::Point2d>& points)
        {
            for (const auto& p : points)
            {
                min_u = std::min(min_u, p.x);
                max_u = std::max(max_u, p.x);
                min_v = std::min(min_v, p.y);
                max_v = std::max(max_v, p.y);
            }
        };

    update_bounds(projected_source);
    update_bounds(projected_target);

    double range_u =
        std::max(max_u - min_u, 1e-6);

    double range_v =
        std::max(max_v - min_v, 1e-6);

    double scale =
        std::min(
            (width - 2.0 * margin) / range_u,
            (height - 2.0 * margin) / range_v
        );

    auto to_pixel =
        [&](const cv::Point2d& p)
        {
            int x =
                cvRound(
                    margin
                    + (p.x - min_u) * scale
                );

            int y =
                cvRound(
                    height
                    - margin
                    - (p.y - min_v) * scale
                );

            return cv::Point(x, y);
        };

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    for (const auto& p : projected_target)
    {
        cv::circle(
            canvas,
            to_pixel(p),
            4,
            cv::Scalar(0, 160, 0),
            -1
        );
    }

    for (const auto& p : projected_source)
    {
        cv::circle(
            canvas,
            to_pixel(p),
            3,
            cv::Scalar(0, 0, 220),
            -1
        );
    }

    cv::putText(
        canvas,
        "Gruen: Ziel",
        cv::Point(20, 30),
        cv::FONT_HERSHEY_SIMPLEX,
        0.7,
        cv::Scalar(0, 120, 0),
        2
    );

    cv::putText(
        canvas,
        "Rot: bewegt",
        cv::Point(20, 60),
        cv::FONT_HERSHEY_SIMPLEX,
        0.7,
        cv::Scalar(0, 0, 180),
        2
    );

    if (!title.empty())
    {
        cv::putText(
            canvas,
            title,
            cv::Point(20, height - 20),
            cv::FONT_HERSHEY_SIMPLEX,
            0.7,
            cv::Scalar(40, 40, 40),
            2
        );
    }

    return canvas;
}

In [ ]:
cv::Mat cloud_before =
    render_point_clouds(
        source_cloud,
        target_cloud,
        "Punktwolken vor ICP"
    );

In [ ]:
show_image(cloud_before)

### Starre Transformation aus bekannten Punktpaaren

Sind die Punktpaare bekannt, kann die beste starre Transformation mit einer SVD bestimmt werden.

ICP verwendet diese Berechnung wiederholt – der entscheidende zusätzliche Schritt ist, dass die Punktkorrespondenzen nach jeder Iteration **neu gesucht** werden.

In [ ]:
struct RigidTransform
{
    cv::Matx33d R;
    cv::Vec3d t;
};

In [ ]:
RigidTransform estimate_rigid_transform(
    const std::vector<cv::Point3d>& source,
    const std::vector<cv::Point3d>& target
)
{
    cv::Vec3d centroid_source(0, 0, 0);
    cv::Vec3d centroid_target(0, 0, 0);

    for (std::size_t i = 0; i < source.size(); ++i)
    {
        centroid_source +=
            cv::Vec3d(
                source[i].x,
                source[i].y,
                source[i].z
            );

        centroid_target +=
            cv::Vec3d(
                target[i].x,
                target[i].y,
                target[i].z
            );
    }

    centroid_source *=
        1.0 / source.size();

    centroid_target *=
        1.0 / target.size();

    cv::Mat H =
        cv::Mat::zeros(
            3,
            3,
            CV_64F
        );

    for (std::size_t i = 0; i < source.size(); ++i)
    {
        cv::Mat a =
            (cv::Mat_<double>(3, 1)
                << source[i].x - centroid_source[0],
                   source[i].y - centroid_source[1],
                   source[i].z - centroid_source[2]);

        cv::Mat b =
            (cv::Mat_<double>(3, 1)
                << target[i].x - centroid_target[0],
                   target[i].y - centroid_target[1],
                   target[i].z - centroid_target[2]);

        H += a * b.t();
    }

    cv::Mat w;
    cv::Mat U;
    cv::Mat Vt;

    cv::SVD::compute(
        H,
        w,
        U,
        Vt
    );

    cv::Mat R =
        Vt.t() * U.t();

    if (cv::determinant(R) < 0.0)
    {
        Vt.row(2) *= -1.0;
        R = Vt.t() * U.t();
    }

    cv::Matx33d R_result;

    for (int r = 0; r < 3; ++r)
    {
        for (int c = 0; c < 3; ++c)
        {
            R_result(r, c) =
                R.at<double>(r, c);
        }
    }

    cv::Vec3d t_result =
        centroid_target
        - R_result * centroid_source;

    RigidTransform result;
    result.R = R_result;
    result.t = t_result;

    return result;
}

In [ ]:
void nearest_neighbors(
    const std::vector<cv::Point3d>& source,
    const std::vector<cv::Point3d>& target,
    std::vector<int>& indices,
    std::vector<double>& distances
)
{
    indices.resize(source.size());
    distances.resize(source.size());

    for (std::size_t i = 0; i < source.size(); ++i)
    {
        double best_distance =
            std::numeric_limits<double>::max();

        int best_index = -1;

        for (std::size_t j = 0; j < target.size(); ++j)
        {
            double distance =
                cv::norm(
                    source[i] - target[j]
                );

            if (distance < best_distance)
            {
                best_distance = distance;
                best_index =
                    static_cast<int>(j);
            }
        }

        indices[i] =
            best_index;

        distances[i] =
            best_distance;
    }
}

In [ ]:
struct ICPResult
{
    std::vector<cv::Point3d> aligned;
    cv::Matx33d R;
    cv::Vec3d t;
    std::vector<double> errors;
};

In [ ]:
ICPResult icp_point_to_point(
    const std::vector<cv::Point3d>& source,
    const std::vector<cv::Point3d>& target,
    int max_iterations = 40,
    double tolerance = 1e-6,
    double max_correspondence_distance =
        std::numeric_limits<double>::infinity()
)
{
    std::vector<cv::Point3d>
        source_current =
            source;

    cv::Matx33d R_total =
        cv::Matx33d::eye();

    cv::Vec3d t_total(
        0.0,
        0.0,
        0.0
    );

    double previous_error =
        std::numeric_limits<double>::infinity();

    std::vector<double>
        error_history;

    for (int iteration = 0;
         iteration < max_iterations;
         ++iteration)
    {
        std::vector<int> indices;
        std::vector<double> distances;

        nearest_neighbors(
            source_current,
            target,
            indices,
            distances
        );

        std::vector<cv::Point3d>
            source_valid;

        std::vector<cv::Point3d>
            target_valid;

        double error_sum = 0.0;

        for (std::size_t i = 0;
             i < source_current.size();
             ++i)
        {
            if (
                distances[i]
                < max_correspondence_distance
            )
            {
                source_valid.push_back(
                    source_current[i]
                );

                target_valid.push_back(
                    target[indices[i]]
                );

                error_sum +=
                    distances[i];
            }
        }

        if (source_valid.size() < 3)
        {
            break;
        }

        RigidTransform delta =
            estimate_rigid_transform(
                source_valid,
                target_valid
            );

        for (auto& point : source_current)
        {
            cv::Vec3d p(
                point.x,
                point.y,
                point.z
            );

            cv::Vec3d transformed =
                delta.R * p
                + delta.t;

            point = cv::Point3d(
                transformed[0],
                transformed[1],
                transformed[2]
            );
        }

        R_total =
            delta.R * R_total;

        t_total =
            delta.R * t_total
            + delta.t;

        double mean_error =
            error_sum
            / source_valid.size();

        error_history.push_back(
            mean_error
        );

        if (
            std::abs(
                previous_error
                - mean_error
            ) < tolerance
        )
        {
            break;
        }

        previous_error =
            mean_error;
    }

    ICPResult result;
    result.aligned = source_current;
    result.R = R_total;
    result.t = t_total;
    result.errors = error_history;

    return result;
}

In [ ]:
ICPResult icp_result =
    icp_point_to_point(
        source_cloud,
        target_cloud,
        40
    );

std::cout << "ICP-Iterationen: "
          << icp_result.errors.size()
          << std::endl;

std::cout << "Letzter mittlerer Punktabstand: "
          << icp_result.errors.back()
          << std::endl;

cv::Mat cloud_after =
    render_point_clouds(
        icp_result.aligned,
        target_cloud,
        "Punktwolken nach ICP"
    );

In [ ]:
show_image(cloud_after)

### ICP-Konvergenz

Auch die Fehlerentwicklung wird im C++-Notebook als OpenCV-Bild dargestellt.

> **Binder-/Notebook-Hinweis:** In der Python-Version übernimmt Matplotlib das Diagramm. Hier zeichnen wir dieselben Werte direkt mit OpenCV.

In [ ]:
cv::Mat render_error_history(
    const std::vector<double>& values
)
{
    const int width = 760;
    const int height = 420;
    const int left = 70;
    const int right = 30;
    const int top = 30;
    const int bottom = 60;

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    if (values.empty())
    {
        return canvas;
    }

    double max_value =
        *std::max_element(
            values.begin(),
            values.end()
        );

    max_value =
        std::max(
            max_value,
            1e-12
        );

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    for (std::size_t i = 0;
         i < values.size();
         ++i)
    {
        double x_fraction =
            values.size() > 1
                ? static_cast<double>(i)
                    / (values.size() - 1)
                : 0.0;

        double y_fraction =
            values[i] / max_value;

        cv::Point p(
            cvRound(
                left
                + x_fraction
                * (width - left - right)
            ),
            cvRound(
                height
                - bottom
                - y_fraction
                * (height - top - bottom)
            )
        );

        cv::circle(
            canvas,
            p,
            4,
            cv::Scalar(200, 80, 0),
            -1
        );

        if (i > 0)
        {
            double prev_x_fraction =
                values.size() > 1
                    ? static_cast<double>(i - 1)
                        / (values.size() - 1)
                    : 0.0;

            double prev_y_fraction =
                values[i - 1] / max_value;

            cv::Point prev(
                cvRound(
                    left
                    + prev_x_fraction
                    * (width - left - right)
                ),
                cvRound(
                    height
                    - bottom
                    - prev_y_fraction
                    * (height - top - bottom)
                )
            );

            cv::line(
                canvas,
                prev,
                p,
                cv::Scalar(200, 80, 0),
                2
            );
        }
    }

    cv::putText(
        canvas,
        "Iteration",
        cv::Point(
            width / 2 - 40,
            height - 20
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.7,
        cv::Scalar(0, 0, 0),
        2
    );

    cv::putText(
        canvas,
        "ICP-Konvergenz",
        cv::Point(20, 25),
        cv::FONT_HERSHEY_SIMPLEX,
        0.7,
        cv::Scalar(0, 0, 0),
        2
    );

    return canvas;
}

cv::Mat error_plot =
    render_error_history(
        icp_result.errors
    );

In [ ]:
show_image(error_plot)

### Warum benötigt ICP eine gute Initialpose?

ICP sucht immer den **aktuell nächsten Punkt**. Bei einer schlechten Startpose können daher bereits die ersten Korrespondenzen falsch sein.

Das Verfahren kann dann zu einem falschen lokalen Minimum konvergieren.

Typische Verbesserungen sind:

- maximale Korrespondenzdistanz,
- robuste Fehlerfunktionen,
- Downsampling,
- Verwendung von Oberflächennormalen,
- gute Initialpose aus einem anderen Pose-Verfahren.

### Mini-Aufgabe: ICP-Initialisierung

Verändern Sie die Rotation und Translation, mit denen die bewegte Punktwolke erzeugt wird.

Testen Sie beispielsweise deutlich größere Rotationen.

Beobachten Sie:

- Wann konvergiert ICP weiterhin zur richtigen Ausrichtung?
- Wann entstehen falsche Korrespondenzen?
- Warum ist ICP eher ein **Pose-Refinement** als ein globales Pose-Schätzverfahren?

In [ ]:
[]()
{
    // TODO: Initiale Fehlpose verändern

    double angle_test =
        25.0 * CV_PI / 180.0;

    cv::Matx33d R_start_test(
        std::cos(angle_test), -std::sin(angle_test), 0.0,
        std::sin(angle_test),  std::cos(angle_test), 0.0,
        0.0,                   0.0,                  1.0
    );

    cv::Vec3d t_start_test(
        0.25,
        -0.15,
        0.10
    );

    std::vector<cv::Point3d>
        source_test;

    for (const auto& point : target_cloud)
    {
        cv::Vec3d p(
            point.x,
            point.y,
            point.z
        );

        cv::Vec3d transformed =
            R_start_test * p
            + t_start_test;

        source_test.emplace_back(
            transformed[0],
            transformed[1],
            transformed[2]
        );
    }

    ICPResult result_test =
        icp_point_to_point(
            source_test,
            target_cloud,
            50
        );

    std::cout
        << "Iterationen: "
        << result_test.errors.size()
        << std::endl;

    std::cout
        << "Letzter mittlerer Punktabstand: "
        << result_test.errors.back()
        << std::endl;

    cv::Mat result_image =
        render_point_clouds(
            result_test.aligned,
            target_cloud,
            "ICP nach veraenderter Initialisierung"
        );

    return show_image(result_image);
}()

## 12. Point-to-Point vs. Point-to-Plane ICP

Die bisherige Implementierung minimiert den direkten Punktabstand:

$e_i =
\|
\mathbf{q}_i-
(\mathbf{R}\mathbf{p}_i+\mathbf{t})
\|$.

Bei **Point-to-Plane ICP** wird stattdessen der Fehler entlang der Oberflächennormalen $\mathbf{n}_i$ betrachtet:

$e_i =
\mathbf{n}_i^T
\left(
\mathbf{q}_i-
(\mathbf{R}\mathbf{p}_i+\mathbf{t})
\right)$.

Point-to-Plane kann besonders auf glatten Oberflächen schneller konvergieren, benötigt dafür aber zusätzlich zuverlässige **Oberflächennormalen**.

Die praktische Implementierung bleibt in diesem Vorbereitungsnotebook bei Point-to-Point ICP.

## 13. Wo kommen gelernte Verfahren ins Spiel?

Die klassische Pipeline lautet vereinfacht:

**Merkmale → Korrespondenzen → PnP → Pose**

Gelernte Verfahren können unterschiedliche Teile dieser Pipeline ersetzen oder ergänzen:

| Teil | Klassisch | Gelernt / modern |
|---|---|---|
| Merkmale | SIFT, ORB | gelernte Features und Deskriptoren |
| Korrespondenzen | Descriptor Matching | gelernte 2D–2D- oder 2D–3D-Zuordnungen |
| Pose | PnP | direkte Pose-Vorhersage |
| Hypothesen | RANSAC | gelernte Bewertung |
| Refinement | PnP-Optimierung, ICP | gelernte Pose-Korrektur / Render-and-Compare |

Wichtig ist daher nicht nur die Frage **„klassisch oder gelernt?“**, sondern:

> **Welcher Teil der Pose-Pipeline wird gelernt und welche geometrische Struktur bleibt erhalten?**

### Render-and-Compare

Bei Render-and-Compare wird aus

**CAD-Modell + Pose-Hypothese**

eine erwartete Ansicht des Objekts erzeugt.

Diese wird mit dem realen Kamerabild verglichen. Ein gelerntes Modell kann anschließend schätzen, wie die Pose verändert werden muss.

Die Grundidee ist damit weiterhin eng mit dem Reprojektionsgedanken verbunden:

**Pose → erwartete Beobachtung → Vergleich → Pose-Korrektur**

### FoundationPose

FoundationPose zielt auf die 6D-Pose-Schätzung und das Tracking **neuer Objektinstanzen**.

Als Objektinformation können beispielsweise

- ein CAD-Modell oder
- Referenzbilder

verwendet werden.

Zur Laufzeit soll keine objektspezifische Nachtrainierung notwendig sein.

Für dieses Vorbereitungsnotebook bleibt FoundationPose bewusst **konzeptionell**.

### Instanz-Level vs. Category-Level

Bei **Instanz-Level Pose Estimation** ist ein konkretes Objekt bekannt.

Beispiel:

**dieses Werkstück → 6D-Pose**

Bei **Category-Level Pose Estimation** soll dagegen auf neue Instanzen einer bekannten Objektkategorie generalisiert werden.

Beispiel:

**beliebige Tasse → 6D-Pose und eventuell Größe**

NOCS verwendet dafür die Idee eines normalisierten kanonischen Objektraums:

**Bildpunkt → Punkt im kanonischen Objektraum → Pose und Skalierung**

## 14. Selbstcheck

Beantworten Sie die Fragen zunächst ohne in die Vorlesungsunterlagen zu schauen.

1. Welche sechs Freiheitsgrade beschreibt eine 6D-Pose?
2. Was bedeutet ${}^{c}\mathbf{T}_{o}$?
3. Welche Informationen benötigt PnP?
4. Was passiert bei der Transformation $\mathbf{P}^c=\mathbf{R}\mathbf{P}+\mathbf{t}$?
5. Warum taucht bei der perspektivischen Projektion die Tiefe $Z$ im Nenner auf?
6. Was beschreibt der Reprojektionsfehler?
7. Welche Größe besitzt der Pose-Jacobian eines einzelnen Bildpunktes?
8. Warum wird die Pose-Optimierung iterativ durchgeführt?
9. Was bewirkt Levenberg–Marquardt?
10. Was unterscheidet Inlier und Outlier?
11. Was ist die Grundidee von RANSAC?
12. Welche Qualitätswerte sollten nach einer RANSAC-Pose betrachtet werden?
13. Was unterscheidet ICP von PnP?
14. Warum benötigt ICP typischerweise eine gute Initialpose?
15. Was unterscheidet Point-to-Point von Point-to-Plane ICP?
16. Müssen gelernte Pose-Verfahren die gesamte klassische Pipeline ersetzen?
17. Was unterscheidet Instanz-Level von Category-Level Pose Estimation?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Drei Translationen und drei Rotationsfreiheitsgrade.
2. Die Transformation vom Objekt- in das Kamerakoordinatensystem.
3. Bekannte 3D-Punkte, zugehörige 2D-Bildpunkte und intrinsische Kameraparameter.
4. Ein Objektpunkt wird mit der aktuellen Pose in das Kamerakoordinatensystem transformiert.
5. Wegen der perspektivischen Projektion: gleiche räumliche Bewegung wirkt bei kleiner Tiefe stärker im Bild.
6. Den Abstand zwischen gemessenem und aus der Pose vorhergesagtem Bildpunkt.
7. $2\times6$.
8. Weil Projektion und Rotation ein nichtlineares Optimierungsproblem bilden.
9. Es berechnet gedämpfte lokale Updates zur Verringerung des Fehlers.
10. Inlier passen geometrisch zum Modell, Outlier nicht.
11. Viele Pose-Hypothesen aus kleinen Teilmengen erzeugen und die Hypothese mit dem größten Konsens auswählen.
12. Inlier-Anzahl, Inlier-Anteil, Reprojektionsfehler und geometrische Plausibilität.
13. PnP nutzt 2D–3D-Korrespondenzen; ICP richtet zwei 3D-Punktmengen aneinander aus.
14. Falsche nächste Nachbarn bei schlechter Initialisierung können zu einem falschen lokalen Minimum führen.
15. Point-to-Point minimiert direkten Punktabstand, Point-to-Plane den Abstand entlang der Oberflächennormalen.
16. Nein. Sie können einzelne Schritte ersetzen oder ergänzen.
17. Instanz-Level betrachtet ein konkretes bekanntes Objekt; Category-Level generalisiert auf neue Instanzen einer bekannten Kategorie.

</details>

## 15. Take-away

Für die Präsenz-LV sollten Sie folgende Punkte mitnehmen:

- Pose Estimation verbindet Bildinformation mit einer räumlichen Transformation.
- PnP berechnet $\mathbf{R}$ und $\mathbf{t}$ aus 2D–3D-Korrespondenzen.
- Die geschätzte Pose kann direkt als Objektkoordinatensystem im Kamerabild visualisiert werden.
- Die Pose transformiert Objektpunkte zunächst in das Kamerakoordinatensystem.
- Die perspektivische Projektion erzeugt daraus vorhergesagte Bildpunkte.
- Der Reprojektionsfehler bildet die Grundlage für Bewertung und Optimierung.
- Jacobians beschreiben lokal, wie eine kleine Poseänderung die Projektion beeinflusst.
- Levenberg–Marquardt kann eine vorhandene Pose iterativ verfeinern.
- RANSAC macht PnP robuster gegenüber falschen Korrespondenzen.
- ICP registriert zwei 3D-Punktmengen und eignet sich besonders zur Pose-Verfeinerung.
- ICP ist lokal und benötigt daher typischerweise eine brauchbare Initialpose.
- Gelernte Verfahren können Merkmale, Korrespondenzen, Hypothesen oder Pose-Refinement übernehmen.
- FoundationPose und Category-Level-Verfahren erweitern den Anwendungsbereich auf neue Objekte bzw. neue Objektinstanzen.

### Robotische Gesamtpipeline

**Kamerabild → Objekterkennung → Pose Estimation → Zielpose → Roboterbewegung**

Damit bildet Pose Estimation eine zentrale Verbindung zwischen **Wahrnehmung und Aktion**.